# KV Cache Efficiency Experiment Harness (Ollama + Pluggable Methods)

This notebook gives you a **modifiable framework** to compare KV-cache-related strategies:

- `fullkv` (no compression; baseline)
- `commonkv`
- `minicache`
- `thinkk`
- `palu`

It is designed for rapid experimentation using your local **Ollama** models.

> Important: Ollama's public API does not directly expose low-level KV-cache internals for arbitrary methods in every backend. This notebook provides:
> 1) a ready-to-run baseline pipeline,
> 2) a consistent adapter interface,
> 3) hooks to wire in real method implementations (Python wrappers, custom inference servers, or modified runtimes).

## 1) Setup

If needed, uncomment and run installation lines.

In [ ]:
# !pip install -U pandas numpy matplotlib requests tqdm

import json
import math
import re
import subprocess
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, List, Optional

import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm

## 2) Discover local Ollama models

In [ ]:
def ollama_list_models() -> List[str]:
    """Return locally available model tags from `ollama list`.

    Works with common table output format where first column is NAME.
    """
    out = subprocess.check_output(["ollama", "list"], text=True)
    lines = [ln.strip() for ln in out.splitlines() if ln.strip()]
    if len(lines) <= 1:
        return []

    models = []
    for ln in lines[1:]:
        parts = re.split(r"\s{2,}", ln)
        if parts:
            models.append(parts[0].strip())
    return models


try:
    LOCAL_MODELS = ollama_list_models()
except Exception as e:
    LOCAL_MODELS = []
    print(f"Could not query ollama list: {e}")

print("Local Ollama models:")
for m in LOCAL_MODELS:
    print(" -", m)

## 3) Core experiment abstractions

In [ ]:
@dataclass
class GenerationResult:
    model: str
    method: str
    prompt_id: str
    prompt_tokens_est: int
    output_text: str
    latency_s: float
    output_tokens_est: int
    options: Dict[str, Any] = field(default_factory=dict)
    meta: Dict[str, Any] = field(default_factory=dict)


class KVCompressionMethod:
    """Base class for KV strategy adapters."""

    name = "base"

    def preprocess_prompt(self, prompt: str, config: Dict[str, Any]) -> str:
        return prompt

    def ollama_options(self, config: Dict[str, Any]) -> Dict[str, Any]:
        return {}

    def invoke(self, model: str, prompt: str, config: Dict[str, Any]) -> Dict[str, Any]:
        """Default invoke uses Ollama /api/generate."""
        url = config.get("ollama_url", "http://localhost:11434/api/generate")
        payload = {
            "model": model,
            "prompt": prompt,
            "stream": False,
            "options": self.ollama_options(config),
        }
        r = requests.post(url, json=payload, timeout=config.get("timeout_s", 300))
        r.raise_for_status()
        return r.json()


def estimate_tokens(text: str) -> int:
    if not text:
        return 0
    return max(1, math.ceil(len(text) / 4))

## 4) Built-in method adapters (editable placeholders + hooks)

In [ ]:
class FullKV(KVCompressionMethod):
    name = "fullkv"


class CommonKV(KVCompressionMethod):
    name = "commonkv"

    def preprocess_prompt(self, prompt: str, config: Dict[str, Any]) -> str:
        keep_ratio = config.get("commonkv_keep_ratio", 0.8)
        n_keep = max(1, int(len(prompt) * keep_ratio))
        return prompt[:n_keep]


class MiniCache(KVCompressionMethod):
    name = "minicache"

    def preprocess_prompt(self, prompt: str, config: Dict[str, Any]) -> str:
        head = config.get("minicache_head_chars", 2000)
        tail = config.get("minicache_tail_chars", 2000)
        if len(prompt) <= head + tail:
            return prompt
        return prompt[:head] + "
... [MINICACHE COMPRESSED] ...
" + prompt[-tail:]


class ThinkK(KVCompressionMethod):
    name = "thinkk"

    def ollama_options(self, config: Dict[str, Any]) -> Dict[str, Any]:
        return {
            "temperature": config.get("thinkk_temperature", 0.2),
            "num_ctx": config.get("thinkk_num_ctx", config.get("num_ctx", 8192)),
        }


class PaLu(KVCompressionMethod):
    name = "palu"

    def invoke(self, model: str, prompt: str, config: Dict[str, Any]) -> Dict[str, Any]:
        palu_url = config.get("palu_url")
        if palu_url:
            payload = {
                "model": model,
                "prompt": prompt,
                "method": "palu",
                "params": config.get("palu_params", {}),
            }
            r = requests.post(palu_url, json=payload, timeout=config.get("timeout_s", 300))
            r.raise_for_status()
            return r.json()
        return super().invoke(model, prompt, config)


METHODS = {
    "fullkv": FullKV(),
    "commonkv": CommonKV(),
    "minicache": MiniCache(),
    "minichache": MiniCache(),  # alias for typo/alternate spelling
    "thinkk": ThinkK(),
    "palu": PaLu(),
}

print("Available method adapters:", sorted(METHODS.keys()))

## 5) Experiment runner

In [ ]:
def run_one(model: str, method_name: str, prompt_id: str, prompt: str, config: Dict[str, Any]) -> GenerationResult:
    method = METHODS[method_name]

    transformed_prompt = method.preprocess_prompt(prompt, config)
    t0 = time.perf_counter()
    resp = method.invoke(model=model, prompt=transformed_prompt, config=config)
    latency = time.perf_counter() - t0

    output_text = resp.get("response", "")

    return GenerationResult(
        model=model,
        method=method_name,
        prompt_id=prompt_id,
        prompt_tokens_est=estimate_tokens(transformed_prompt),
        output_text=output_text,
        latency_s=latency,
        output_tokens_est=estimate_tokens(output_text),
        options=method.ollama_options(config),
        meta={"raw": resp},
    )


def run_grid(prompts: Dict[str, str], models: List[str], methods: List[str], config: Optional[Dict[str, Any]] = None) -> pd.DataFrame:
    config = config or {}
    rows = []

    total = len(prompts) * len(models) * len(methods)
    pbar = tqdm(total=total, desc="KV experiments")

    for model in models:
        for pid, prompt in prompts.items():
            for method in methods:
                try:
                    res = run_one(model, method, pid, prompt, config)
                    rows.append({
                        "model": res.model,
                        "method": res.method,
                        "prompt_id": res.prompt_id,
                        "prompt_tokens_est": res.prompt_tokens_est,
                        "output_tokens_est": res.output_tokens_est,
                        "latency_s": res.latency_s,
                        "tok_per_s_est": (res.output_tokens_est / res.latency_s) if res.latency_s else np.nan,
                        "output_preview": res.output_text[:200],
                    })
                except Exception as e:
                    rows.append({
                        "model": model,
                        "method": method,
                        "prompt_id": pid,
                        "prompt_tokens_est": np.nan,
                        "output_tokens_est": np.nan,
                        "latency_s": np.nan,
                        "tok_per_s_est": np.nan,
                        "output_preview": f"ERROR: {e}",
                    })
                pbar.update(1)

    return pd.DataFrame(rows)

## 6) Define prompts/models and run

In [ ]:
SELECTED_MODELS = LOCAL_MODELS[:2] if LOCAL_MODELS else ["llama3.1:8b"]

PROMPTS = {
    "long_qa": """You are given a long document. Summarize key findings and list open research questions.

Document:
""" + ("Efficient transformers rely on memory hierarchy optimization. " * 600),
    "code_reasoning": """Review this pseudo-code and propose micro-optimizations and complexity tradeoffs:
""" + ("for i in range(n): for j in range(m): process(i, j)\n" * 250),
}

METHOD_LIST = ["fullkv", "commonkv", "minicache", "thinkk", "palu"]

CONFIG = {
    "ollama_url": "http://localhost:11434/api/generate",
    "timeout_s": 300,
    "num_ctx": 8192,
    "commonkv_keep_ratio": 0.75,
    "minicache_head_chars": 2400,
    "minicache_tail_chars": 1600,
    "thinkk_num_ctx": 4096,
    "thinkk_temperature": 0.1,
    # "palu_url": "http://localhost:8000/generate",
    # "palu_params": {"rank": 64, "window": 256},
}

results_df = run_grid(
    prompts=PROMPTS,
    models=SELECTED_MODELS,
    methods=METHOD_LIST,
    config=CONFIG,
)

results_df.head(20)

## 7) Aggregate metrics

In [ ]:
summary = (
    results_df
    .groupby(["model", "method"], as_index=False)
    .agg(
        runs=("latency_s", "count"),
        latency_mean_s=("latency_s", "mean"),
        latency_median_s=("latency_s", "median"),
        out_tok_mean=("output_tokens_est", "mean"),
        tps_mean=("tok_per_s_est", "mean"),
    )
    .sort_values(["model", "latency_mean_s"], ascending=[True, True])
)

summary

## 8) Plot latency and throughput

In [ ]:
import matplotlib.pyplot as plt

for model in summary["model"].unique():
    sub = summary[summary["model"] == model]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].bar(sub["method"], sub["latency_mean_s"])
    axes[0].set_title(f"{model}: mean latency")
    axes[0].set_ylabel("seconds")
    axes[0].tick_params(axis="x", rotation=45)

    axes[1].bar(sub["method"], sub["tps_mean"])
    axes[1].set_title(f"{model}: estimated throughput")
    axes[1].set_ylabel("tokens/sec (estimated)")
    axes[1].tick_params(axis="x", rotation=45)

    plt.tight_layout()
    plt.show()

## 9) Save raw outputs for later analysis

In [ ]:
out_dir = Path("artifacts")
out_dir.mkdir(exist_ok=True)

csv_path = out_dir / "kv_experiment_results.csv"
summary_path = out_dir / "kv_experiment_summary.csv"

results_df.to_csv(csv_path, index=False)
summary.to_csv(summary_path, index=False)

print("Saved:")
print(" -", csv_path)
print(" -", summary_path)

## 10) Where to plug in your new idea

Best extension points:

1. Create a new subclass of `KVCompressionMethod`.
2. Implement one or more of:
   - `preprocess_prompt` (quick simulation/proxy)
   - `ollama_options` (runtime knobs)
   - `invoke` (real integration with your backend/research runtime)
3. Add it to `METHODS` and include the name in `METHOD_LIST`.

This lets you compare your method against `fullkv`, `commonkv`, `minicache`, `thinkk`, and `palu` under a unified evaluation loop.